In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Step 2: Import packages
from openai import OpenAI
import json
import time
import uuid
import os

In [ ]:
os.environ['OPENAI_API_KEY'] = "..."
client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY"),
)

In [ ]:
# Step 4: Output directory in your Google Drive
output_dir = "/content/drive/MyDrive/parenting_study"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "parenting_study_posts_streamed_batched_GPT41.json")

In [ ]:
# Step 5: Generation settings
POSTS_PER_BATCH = 10
TOTAL_POSTS = 200
BATCHES = TOTAL_POSTS // POSTS_PER_BATCH  # 20 batches

# 2x2 design: 4 conditions × 5 batches each
design = [
    {"condition": "treatment", "tone": "narrative"},
    {"condition": "treatment", "tone": "factual"},
    {"condition": "control", "tone": "narrative"},
    {"condition": "control", "tone": "factual"},
]

batch_plan = []
for group in design:
    for i in range(5):
        batch_plan.append({
            "condition": group["condition"],
            "tone": group["tone"],
            "batch_id": f"{group['condition']}_{group['tone']}_batch{i+1}"
        })
# Prompt generator using your original language
def make_prompt(batch_id, condition, tone, n_posts=10):
    return f"""
You are an expert content creator and behavioral researcher. Your task is to generate {n_posts} realistic social media-style posts for a study on gendered parenting burdens and perspective taking. The posts will simulate Reddit discussions from r/Mommit, where mothers share their parenting experiences. These texts will be used in a 2x2 full factorial experimental design and for latent treatment effect modeling using supervised machine learning (e.g., supervised Indian buffet process per Fong & Grimmer, 2016).

Study Context
The study investigates whether targeted perspective-taking—reading first-person narratives from mothers—can increase empathy and shift attitudes toward more equitable parenting. Although both parents are caregivers, gender norms often place disproportionate emotional and logistical labor on mothers (e.g., Hochschild & Machung, 1989; Daminger, 2019).

Experimental Design
This batch corresponds to the following condition and tone:
- Condition: {"Treatment – Challenges and invisible burdens" if condition == "treatment" else "Control – Joys and positive parenting moments"}
- Tone: {"Narrative – Personal stories" if tone == "narrative" else "Factual – Descriptive, observational"}

Post Specifications
- Each post should contain approximately 7 sentences, each with 15–20 words.
- Posts should sound authentic for Reddit (casual but thoughtful).
- Each post should be followed by 3 realistic comments as they might appear in r/Mommit—supportive or relatable, or questioning.
- All content must be written in first-person perspective and should exhibit diverse latent themes, allowing downstream ML models to identify meaningful variation across posts.

Variation Goals (for ML analysis)
Posts should vary along multiple dimensions beyond the 2x2 design:
- Degree of emotional intensity (from mild frustration to emotional exhaustion or deep joy)
- Focus on individual vs systemic or cultural issues
- Themes covered (e.g., planning meals, bedtime routines, childcare costs, role expectations, milestones)
- Social class implications (implicitly or explicitly)
- Partner involvement (or lack thereof)
- Tone and attitude (resigned, hopeful, angry, content, grateful, etc.)

This variation will allow us to infer low-dimensional latent treatments through supervised machine learning approaches.

Output Format
Return each post as a single-line JSON object (NDJSON format), one per line.
Each object must include:
- post_id: Unique identifier for the post
- post_text: The full post body (~7 sentences)
- comment_1, comment_2, comment_3: Three realistic user replies
- condition: "treatment" or "control"
- tone: "narrative" or "factual"

Return exactly {n_posts} posts. Do not wrap the output in an array. Only valid JSON objects, one per line.
"""

# Streaming and parsing NDJSON from API
def generate_batch(prompt):
    # Use the newer API version (requires openai>=1.x)
    response = client.responses.create(
        model="gpt-4.1",
        instructions="You are an expert content creator and behavioral researcher.",
        input=prompt,
        temperature=0.8    )

    content = response.output_text

    posts = []
    for line in content.strip().split("\n"):
        if not line.strip():
            continue
        try:
            post = json.loads(line.strip())
            if "post_id" not in post:
                post["post_id"] = str(uuid.uuid4())
            posts.append(post)
            print(f"✅ Parsed post: {post['post_id']}")
        except json.JSONDecodeError:
            print(f"❌ Failed to parse line:\n{line.strip()}")
    return posts

# Main loop
all_posts = []

for i, batch in enumerate(batch_plan):
    print(f"\n🚀 Generating batch {i+1}/{len(batch_plan)}: {batch['batch_id']}")
    prompt = make_prompt(
        batch_id=batch["batch_id"],
        condition=batch["condition"],
        tone=batch["tone"],
        n_posts=POSTS_PER_BATCH
    )

    try:
        posts = generate_batch(prompt)
        all_posts.extend(posts)
    except Exception as e:
        print(f"❌ Error in batch {batch['batch_id']}: {e}")

    time.sleep(2)

# Save final JSON
with open(output_path, "w") as f:
    json.dump(all_posts, f, indent=2)

print(f"\n✅ All done! Total posts generated: {len(all_posts)}")
print(f"📁 Saved to: {output_path}")


🚀 Generating batch 1/20: treatment_narrative_batch1
✅ Parsed post: p1
✅ Parsed post: p2
✅ Parsed post: p3
✅ Parsed post: p4
✅ Parsed post: p5
✅ Parsed post: p6
✅ Parsed post: p7
✅ Parsed post: p8
✅ Parsed post: p9
✅ Parsed post: p10

🚀 Generating batch 2/20: treatment_narrative_batch2
✅ Parsed post: 1
✅ Parsed post: 2
✅ Parsed post: 3
✅ Parsed post: 4
✅ Parsed post: 5
✅ Parsed post: 6
✅ Parsed post: 7
✅ Parsed post: 8
✅ Parsed post: 9
✅ Parsed post: 10

🚀 Generating batch 3/20: treatment_narrative_batch3
✅ Parsed post: post_001
✅ Parsed post: post_002
✅ Parsed post: post_003
✅ Parsed post: post_004
✅ Parsed post: post_005
✅ Parsed post: post_006
✅ Parsed post: post_007
✅ Parsed post: post_008
✅ Parsed post: post_009
✅ Parsed post: post_010

🚀 Generating batch 4/20: treatment_narrative_batch4
✅ Parsed post: p001
✅ Parsed post: p002
✅ Parsed post: p003
✅ Parsed post: p004
✅ Parsed post: p005
✅ Parsed post: p006
✅ Parsed post: p007
✅ Parsed post: p008
✅ Parsed post: p009
✅ Parsed post: p